# Sparse Release Drawdown-Control Fold-Local BC Warm Start v1

## Purpose

The previous checkpoint-objective screen showed that no existing candidate clears the drawdown-control targets and the `+30pp / <=15% MaxDD` gate. This notebook moves one step closer to training without turning the failed 2022 evaluation interval into direct labels.

It uses the failed 2022 interval only as a **state prototype**. Labels are generated only on non-overlapping `wf_2022_train` dates (`<= 2020-12-31`) that look similar to that prototype. The run then checks action-surface health and trains a diagnostic BC warm-start actor if, and only if, the labels are fold-local and non-degenerate.

## Experiment Governance

- **Baselines**: `bc_initialization` and `ppo_selected` from `sparse_release_failed_fold_interval_diagnostic_v1`; candidate objective scores from `sparse_release_drawdown_control_checkpoint_score_v1`.
- **Role**: `diagnose` / training-readiness handoff.
- **Changed layer**: `model_selection`, `training_objective`, `model`.
- **Target window**: `wf_2022_train` label generation; future validation must replay `wf_2022` and `wf_2023_2024` before promotion.
- **Acceptance**: fold-local scope gate, BC readiness gate, downtrend slice fit gate, and no runtime promotion from this notebook alone.
- **Failure mode**: label collapse, eval leakage, or BC slice overfit blocks PPO handoff.

## External Basis

- Risk-sensitive MDP / CVaR objective: https://arxiv.org/abs/1506.02188
- Deep portfolio RL allocation as sequential risk control: https://arxiv.org/abs/1706.10059
- Learning from demonstrations / BC warm start before RL: https://arxiv.org/abs/1704.03732
- Qlib reproducible research boundary: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL data/environment/agent/evaluation separation: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- PBO / CSCV overfit warning: https://doi.org/10.21314/JCF.2016.322

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
import sys

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'AGENTS.md').exists() and (path / 'src').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    action_distribution_frame,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    summarize_bc_slice_fit,
)
from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveBCReadinessConfig,
    audit_state_objective_target_scope,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_scope_audit,
)
from src.soxl_quant.config import load_profile

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

EXPERIMENT = 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1'
OUT_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
FAILED_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'sparse_release_failed_fold_interval_diagnostic_v1'
TARGET_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'sparse_release_drawdown_control_target_handoff_v1'
SCORE_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'sparse_release_drawdown_control_checkpoint_score_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0
SEED = 42

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
frame = pd.read_pickle(FEATURE_FRAME_PATH).copy()
if 'datetime' not in frame.columns:
    frame = frame.reset_index().rename(columns={'index': 'datetime'})
frame['datetime'] = pd.to_datetime(frame['datetime'])

ppo_daily = pd.read_csv(FAILED_DIR / 'ppo_selected_daily_2022.csv', parse_dates=['datetime'])
intervals = pd.read_csv(FAILED_DIR / 'ppo_worst_drawdown_intervals.csv')
feature_focus = pd.read_csv(TARGET_DIR / 'drawdown_interval_feature_focus.csv')
source_targets = pd.read_csv(TARGET_DIR / 'drawdown_control_training_targets.csv', parse_dates=['datetime'])
checkpoint_summary = json.loads((SCORE_DIR / 'summary.json').read_text())

interval = intervals.iloc[0]
interval_start = pd.Timestamp(interval['start'])
interval_end = pd.Timestamp(interval['end'])
top_features = [
    feature for feature in feature_focus['feature'].head(5).tolist()
    if feature in frame.columns and feature in ppo_daily.columns
]
failed_interval = ppo_daily[(ppo_daily['datetime'] >= interval_start) & (ppo_daily['datetime'] <= interval_end)].copy()
prototype = failed_interval[top_features].mean()

risk_source_rows = source_targets[source_targets['is_downtrend_state'].astype(bool)].copy()
risk_teacher_position = float(risk_source_rows['teacher_position'].mean())
risk_target_position = float(risk_source_rows['risk_target_position'].mean())

coverage = {
    'experiment': EXPERIMENT,
    'feature_frame_path': str(FEATURE_FRAME_PATH.relative_to(ROOT)),
    'source_failed_experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
    'source_target_experiment': 'sparse_release_drawdown_control_target_handoff_v1',
    'source_checkpoint_score_experiment': 'sparse_release_drawdown_control_checkpoint_score_v1',
    'prototype_interval_start': str(interval_start.date()),
    'prototype_interval_end': str(interval_end.date()),
    'prototype_features': top_features,
    'risk_teacher_position': risk_teacher_position,
    'risk_target_position': risk_target_position,
    'checkpoint_score_decision': checkpoint_summary['decision'],
}

(OUT_DIR / 'coverage.json').write_text(json.dumps(coverage, indent=2, ensure_ascii=False) + chr(10))
coverage

{'experiment': 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1',
 'feature_frame_path': 'data/soxl_profile_balanced_rule_residual_v22.pkl',
 'source_failed_experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
 'source_target_experiment': 'sparse_release_drawdown_control_target_handoff_v1',
 'source_checkpoint_score_experiment': 'sparse_release_drawdown_control_checkpoint_score_v1',
 'prototype_interval_start': '2022-03-30',
 'prototype_interval_end': '2022-10-14',
 'prototype_features': ['below_ma63_days',
  'anchor_gap_180',
  'negative_gap_frequency_21',
  'below_anchor_ratio_63',
  'rolling_new_low_ratio_63'],
 'risk_teacher_position': 0.4041417322936507,
 'risk_target_position': 0.25,
 'checkpoint_score_decision': 'checkpoint_objective_no_promotion'}

In [3]:
train_start = pd.Timestamp('2011-03-11')
train_end = pd.Timestamp('2020-12-31')
eval_start = pd.Timestamp('2022-01-01')
eval_end = pd.Timestamp('2022-12-31')
fold = {
    'name': 'wf_2022_train',
    'train_start': str(train_start.date()),
    'train_end': str(train_end.date()),
    'eval_start': str(eval_start.date()),
    'eval_end': str(eval_end.date()),
}

train_frame = frame[(frame['datetime'] >= train_start) & (frame['datetime'] <= train_end)].copy()
feature_std = train_frame[top_features].std().replace(0.0, np.nan)
train_frame['prototype_distance'] = (
    ((train_frame[top_features] - prototype) / feature_std)
    .pow(2)
    .mean(axis=1)
    .pow(0.5)
)
train_frame['prototype_similarity'] = 1.0 / (1.0 + train_frame['prototype_distance'])

prototype_payload = {
    'features': top_features,
    'prototype': {key: float(value) for key, value in prototype.to_dict().items()},
    'train_feature_mean': {key: float(value) for key, value in train_frame[top_features].mean().to_dict().items()},
    'train_feature_std': {key: float(value) for key, value in feature_std.to_dict().items()},
}
(OUT_DIR / 'feature_prototype.json').write_text(json.dumps(prototype_payload, indent=2, ensure_ascii=False) + chr(10))

train_frame[['datetime', 'prototype_distance', 'prototype_similarity', *top_features]].nsmallest(5, 'prototype_distance')

,datetime,prototype_distance,prototype_similarity,below_ma63_days,anchor_gap_180,negative_gap_frequency_21,below_anchor_ratio_63,rolling_new_low_ratio_63
1938,2018-11-20,0.369428,0.730232,0.650794,-0.382385,0.476190,0.793651,0.126984
1949,2018-12-07,0.434723,0.696999,0.825397,-0.396203,0.571429,0.920635,0.142857
1947,2018-12-04,0.438321,0.695255,0.793651,-0.313315,0.523810,0.920635,0.126984
1951,2018-12-11,0.442496,0.693243,0.857143,-0.355873,0.523810,0.936508,0.142857
1944,2018-11-29,0.455294,0.687147,0.746032,-0.291961,0.476190,0.873016,0.126984


In [4]:
def make_foldlocal_targets(similar_rows: pd.DataFrame, *, quantile: float) -> pd.DataFrame:
    rows = similar_rows.copy().sort_values('datetime')
    max_distance = float(rows['prototype_distance'].max()) if len(rows) else 0.0
    weights = 1.0 + (max_distance - rows['prototype_distance']).clip(lower=0.0) * 5.0
    targets = pd.DataFrame(
        {
            'datetime': rows['datetime'].to_numpy(),
            'candidate': 'train_analog_drawdown_control',
            'window': fold['name'],
            'fold': fold['name'],
            'target_scope': 'fold_local',
            'train_start': fold['train_start'],
            'train_end': fold['train_end'],
            'eval_start': fold['eval_start'],
            'eval_end': fold['eval_end'],
            'teacher_position': risk_teacher_position,
            'rebound_target_position': risk_teacher_position,
            'risk_target_position': risk_target_position,
            'objective_weight': np.clip(weights, 1.0, 10.0),
            'is_rebound_cluster': False,
            'is_downtrend_state': True,
            'drawdown_control_state': True,
            'source_quantile': float(quantile),
            'prototype_distance': rows['prototype_distance'].to_numpy(),
            'prototype_similarity': rows['prototype_similarity'].to_numpy(),
        }
    )
    return targets.reset_index(drop=True)

quantiles = [0.02, 0.05, 0.10, 0.15, 0.20]
profile_names = ['risk_budget_usage_v1', 'balanced_rule_residual_v22', 'balanced_rule_residual_v19']
readiness_rows = []
targets_by_quantile = {}
for quantile in quantiles:
    n = max(1, int(len(train_frame) * quantile))
    selected_rows = train_frame.nsmallest(n, 'prototype_distance')
    targets = make_foldlocal_targets(selected_rows, quantile=quantile)
    targets_by_quantile[quantile] = targets
    for profile_name in profile_names:
        profile = load_profile(profile_name)
        dataset = build_state_objective_bc_dataset_from_frame(
            frame,
            targets=targets,
            profile=profile,
            sample_size=0,
        )
        action_count = int(dataset.actions.max()) + 1 if len(dataset.actions) else 0
        dist = action_distribution_frame(dataset.actions, action_count=max(action_count, 1))
        positive = dist[dist['count'] > 0]
        share = positive['share'].to_numpy(dtype=float)
        entropy = float(-(share * np.log(share + 1e-12)).sum()) if len(share) else 0.0
        readiness_rows.append(
            {
                'profile': f'{profile_name}__q{quantile:.2f}',
                'base_profile': profile_name,
                'quantile': float(quantile),
                'rows': int(len(targets)),
                'samples': int(len(dataset.actions)),
                'unique_actions': int((dist['count'] > 0).sum()),
                'dominant_action_share': float(dist['share'].max()) if len(dist) else 1.0,
                'action_entropy': entropy,
                'min_prototype_distance': float(targets['prototype_distance'].min()),
                'max_prototype_distance': float(targets['prototype_distance'].max()),
                'mean_prototype_distance': float(targets['prototype_distance'].mean()),
            }
        )

label_comparison = pd.DataFrame(readiness_rows)
readiness_scores = score_state_objective_bc_readiness(
    label_comparison[['profile', 'samples', 'unique_actions', 'dominant_action_share', 'action_entropy']],
    target_scope='fold_local',
    config=StateObjectiveBCReadinessConfig(
        min_samples=250,
        min_unique_actions=4,
        max_dominant_action_share=0.75,
    ),
)
readiness_scores = readiness_scores.merge(label_comparison, on=['profile', 'samples', 'unique_actions', 'dominant_action_share', 'action_entropy'])
readiness_summary = summarize_state_objective_bc_readiness(readiness_scores)
readiness_scores.to_csv(OUT_DIR / 'bc_readiness_scores.csv', index=False)
label_comparison.to_csv(OUT_DIR / 'bc_label_comparison.csv', index=False)

readiness_scores.sort_values(['bc_readiness_gate', 'quantile', 'bc_readiness_loss'], ascending=[False, True, True])

,profile,samples,unique_actions,dominant_action_share,action_entropy,sample_gate,label_distribution_gate,fold_local_scope_gate,bc_readiness_gate,bc_readiness_loss,target_scope,base_profile,quantile,rows,min_prototype_distance,max_prototype_distance,mean_prototype_distance
1,risk_budget_usage_v1__q0.15,370,4,0.594595,9.032763e-01,True,True,True,True,0.00000,fold_local,risk_budget_usage_v1,0.15,370,0.369428,1.456114,1.055537
0,risk_budget_usage_v1__q0.20,494,4,0.483806,9.323746e-01,True,True,True,True,0.00000,fold_local,risk_budget_usage_v1,0.20,494,0.369428,1.605610,1.171966
4,risk_budget_usage_v1__q0.02,49,3,0.755102,7.125413e-01,False,False,True,False,1.85502,fold_local,risk_budget_usage_v1,0.02,49,0.369428,0.713029,0.582867
13,balanced_rule_residual_v19__q0.02,49,1,1.000000,-1.000089e-12,False,False,True,False,6.30400,fold_local,balanced_rule_residual_v19,0.02,49,0.369428,0.713029,0.582867
14,balanced_rule_residual_v22__q0.02,49,1,1.000000,-1.000089e-12,False,False,True,False,6.30400,fold_local,balanced_rule_residual_v22,0.02,49,0.369428,0.713029,0.582867
3,risk_budget_usage_v1__q0.05,123,3,0.747967,6.883876e-01,False,False,True,False,1.50800,fold_local,risk_budget_usage_v1,0.05,123,0.369428,0.959336,0.734774
11,balanced_rule_residual_v19__q0.05,123,1,1.000000,-1.000089e-12,False,False,True,False,6.00800,fold_local,balanced_rule_residual_v19,0.05,123,0.369428,0.959336,0.734774
12,balanced_rule_residual_v22__q0.05,123,1,1.000000,-1.000089e-12,False,False,True,False,6.00800,fold_local,balanced_rule_residual_v22,0.05,123,0.369428,0.959336,0.734774
2,risk_budget_usage_v1__q0.10,247,3,0.720648,7.148944e-01,False,False,True,False,1.01200,fold_local,risk_budget_usage_v1,0.10,247,0.369428,1.208045,0.909959
9,balanced_rule_residual_v19__q0.10,247,1,1.000000,-1.000089e-12,False,False,True,False,5.51200,fold_local,balanced_rule_residual_v19,0.10,247,0.369428,1.208045,0.909959


In [5]:
passing = readiness_scores[readiness_scores['bc_readiness_gate'].astype(bool)].copy()
if passing.empty:
    selected_row = readiness_scores.sort_values(['bc_readiness_loss', 'quantile']).iloc[0]
else:
    selected_row = passing.sort_values(['quantile', 'mean_prototype_distance']).iloc[0]
selected_profile = str(selected_row['base_profile'])
selected_quantile = float(selected_row['quantile'])
selected_targets = targets_by_quantile[selected_quantile].copy()
selected_targets.to_csv(OUT_DIR / 'selected_foldlocal_targets.csv', index=False)

scope_audit = audit_state_objective_target_scope(selected_targets, folds=[fold])
scope_summary = summarize_state_objective_target_scope_audit(scope_audit)
scope_audit.to_csv(OUT_DIR / 'target_scope_audit.csv', index=False)

selected_profile, selected_quantile, scope_summary, selected_targets.head()

('risk_budget_usage_v1',
 0.15,
 {'decision': 'state_objective_targets_training_ready',
  'training_allowed': True,
  'leakage_gate': True,
  'audited_groups': 1,
  'blocked_groups': 0,
  'eval_overlap_rows': 0,
  'outside_train_eval_rows': 0,
  'scope_issues': [],
  'top_scope_issue': '',
  'next_mechanism': 'run_fold_local_state_objective_training_or_bc'},
     datetime                      candidate         window           fold target_scope train_start   train_end  eval_start    eval_end  teacher_position  rebound_target_position  \
 0 2011-06-28  train_analog_drawdown_control  wf_2022_train  wf_2022_train   fold_local  2011-03-11  2020-12-31  2022-01-01  2022-12-31          0.404142                 0.404142   
 1 2011-06-29  train_analog_drawdown_control  wf_2022_train  wf_2022_train   fold_local  2011-03-11  2020-12-31  2022-01-01  2022-12-31          0.404142                 0.404142   
 2 2011-06-30  train_analog_drawdown_control  wf_2022_train  wf_2022_train   fold_local  2011

In [6]:
profile = load_profile(selected_profile)
dataset = build_state_objective_bc_dataset_from_frame(
    frame,
    targets=selected_targets,
    profile=profile,
    sample_size=0,
)

train_config = StateObjectiveBCTrainingConfig(steps=200, lr=0.001, batch_size=128, seed=SEED)
weight_config = StateObjectiveBCSampleWeightConfig(
    downtrend_multiplier=2.0,
    objective_weight_power=0.5,
    class_balanced_beta=0.99,
    weight_cap=20.0,
)
fit_result = fit_state_objective_bc_actor_from_frame(
    frame,
    targets=selected_targets,
    profile=profile,
    train_config=train_config,
    sample_weight_config=weight_config,
    sample_size=0,
    rng_seed=SEED,
)

slice_metrics = evaluate_bc_actor_slice_fit(
    fit_result.actor,
    dataset.obs,
    dataset.actions,
    dataset.metadata,
    slice_cols=('is_downtrend_state', 'is_rebound_cluster'),
)
slice_summary = summarize_bc_slice_fit(
    slice_metrics,
    required_slices=('is_downtrend_state',),
    min_accuracy=0.60,
    min_samples=100,
)

action_distribution = fit_result.action_distribution.copy()
action_distribution.to_csv(OUT_DIR / 'selected_action_distribution.csv', index=False)
slice_metrics.to_csv(OUT_DIR / 'bc_slice_metrics.csv', index=False)
(OUT_DIR / 'bc_fit_metrics.json').write_text(json.dumps(fit_result.metrics, indent=2, ensure_ascii=False) + chr(10))

torch.save(
    {
        'state_dict': fit_result.actor.state_dict(),
        'profile': selected_profile,
        'experiment': EXPERIMENT,
        'selected_quantile': selected_quantile,
        'train_config': train_config.__dict__,
        'sample_weight_config': weight_config.__dict__,
        'metrics': fit_result.metrics,
    },
    OUT_DIR / 'risk_budget_usage_v1_drawdown_control_bc_actor.pt',
)

fit_result.metrics, action_distribution, slice_metrics, slice_summary

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


({'samples': 370,
  'steps': 200,
  'lr': 0.001,
  'batch_size': 128,
  'before_loss': 1.6095991804793075,
  'before_accuracy': 0.13513513513513514,
  'train_last_loss': 0.9534941911697388,
  'train_last_batch_accuracy': 0.9375,
  'after_loss': 1.0070911304370778,
  'after_accuracy': 0.9,
  'weight_mean': 1.0,
  'weight_min': 0.4027975797653198,
  'weight_max': 6.115694522857666},
    action  count     share
 0       0     10  0.027027
 1       1      0  0.000000
 2       2    125  0.337838
 3       3     15  0.040541
 4       4    220  0.594595,
                 slice  samples  accuracy      loss  label_unique_actions  predicted_unique_actions  label_dominant_action  label_dominant_share  predicted_dominant_action  \
 0                 all      370       0.9  1.007091                     4                         3                      4              0.594595                          4   
 1  is_downtrend_state      370       0.9  1.007091                     4                        

In [7]:
training_allowed = bool(
    selected_row.get('bc_readiness_gate', False)
    and scope_summary.get('training_allowed', False)
    and slice_summary.get('slice_fit_gate', False)
)
selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'training_objective', 'model'],
    'profile': selected_profile,
    'selected_quantile': selected_quantile,
    'decision': 'foldlocal_bc_warmstart_ready_no_promotion' if training_allowed else 'foldlocal_bc_warmstart_blocked_no_promotion',
    'training_allowed': training_allowed,
    'promotion_allowed': False,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'selected_samples': int(selected_row['samples']),
    'selected_unique_actions': int(selected_row['unique_actions']),
    'selected_dominant_action_share': float(selected_row['dominant_action_share']),
    'selected_action_entropy': float(selected_row['action_entropy']),
    'bc_after_accuracy': float(fit_result.metrics['after_accuracy']),
    'bc_after_loss': float(fit_result.metrics['after_loss']),
    'scope_decision': scope_summary['decision'],
    'slice_fit_decision': slice_summary['decision'],
    'next_mechanism': 'run_foldlocal_bc_warmstart_walkforward_ppo_screen' if training_allowed else 'revise_foldlocal_target_or_action_surface_before_training',
    'external_sources': [
        'https://arxiv.org/abs/1506.02188',
        'https://arxiv.org/abs/1706.10059',
        'https://arxiv.org/abs/1704.03732',
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
    'sidecars': [
        'coverage.json',
        'feature_prototype.json',
        'bc_label_comparison.csv',
        'bc_readiness_scores.csv',
        'selected_foldlocal_targets.csv',
        'target_scope_audit.csv',
        'selected_action_distribution.csv',
        'bc_slice_metrics.csv',
        'bc_fit_metrics.json',
        'risk_budget_usage_v1_drawdown_control_bc_actor.pt',
        'selection.json',
        'summary.json',
    ],
}

conclusion = [
    f"Selected `{selected_profile}` at prototype quantile {selected_quantile:.2f}: {selection['selected_samples']} fold-local samples, {selection['selected_unique_actions']} actions, dominant share {selection['selected_dominant_action_share']:.3f}.",
    f"Direct `v22/v19` residual action surfaces collapsed in the readiness screen; `risk_budget_usage_v1` is the non-degenerate surface for this drawdown-control warm start.",
    f"BC fit improved accuracy from {fit_result.metrics['before_accuracy']:.3f} to {fit_result.metrics['after_accuracy']:.3f}; downtrend slice gate is {slice_summary['slice_fit_gate']}.",
    f"Scope audit decision is `{scope_summary['decision']}`; selected rows are <= {fold['train_end']} and do not overlap the 2022 eval window.",
    "No runtime promotion is allowed from BC label fit. Next step is a fold-local PPO/replay screen from this BC warm start, followed by wf_2022 and wf_2023_2024 validation against +30pp / <=15% MaxDD.",
]

summary = {
    'experiment': EXPERIMENT,
    'decision': selection['decision'],
    'training_allowed': training_allowed,
    'promotion_allowed': False,
    'coverage': coverage,
    'readiness_summary': readiness_summary,
    'selected_readiness_row': selected_row.to_dict(),
    'scope_summary': scope_summary,
    'slice_summary': slice_summary,
    'bc_fit_metrics': fit_result.metrics,
    'selection': selection,
    'conclusion': conclusion,
}

(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, ensure_ascii=False) + chr(10))
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False) + chr(10))

summary

{'experiment': 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1',
 'decision': 'foldlocal_bc_warmstart_ready_no_promotion',
 'training_allowed': True,
 'promotion_allowed': False,
 'coverage': {'experiment': 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1',
  'feature_frame_path': 'data/soxl_profile_balanced_rule_residual_v22.pkl',
  'source_failed_experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
  'source_target_experiment': 'sparse_release_drawdown_control_target_handoff_v1',
  'source_checkpoint_score_experiment': 'sparse_release_drawdown_control_checkpoint_score_v1',
  'prototype_interval_start': '2022-03-30',
  'prototype_interval_end': '2022-10-14',
  'prototype_features': ['below_ma63_days',
   'anchor_gap_180',
   'negative_gap_frequency_21',
   'below_anchor_ratio_63',
   'rolling_new_low_ratio_63'],
  'risk_teacher_position': 0.4041417322936507,
  'risk_target_position': 0.25,
  'checkpoint_score_decision': 'checkpoint_objective_no_promot

## Notebook Conclusion

This notebook produces a diagnostic BC warm-start artifact, not a promoted runtime model. Its important guardrail is that the 2022 failed interval is used only to define a feature-state prototype; the actual labels and training samples come from the non-overlapping walk-forward training window. Promotion still requires a future walk-forward PPO/replay screen and fixed-split review.